# Entrega 1 — Análise Comparativa de Representações
**Objetivo:** Comparar BoW, TF-IDF e Embeddings na captura de semântica das manifestações da ouvidoria.


In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# 1. Carregar Dados
with open('manifestacoes.json', 'r', encoding='utf-8') as f:
    dados = json.load(f)

df = pd.DataFrame(dados)
textos = df['texto'].tolist()
ids = df['id'].tolist()

print(f"Total de manifestações carregadas: {len(df)}")


In [ ]:
# 2. Gerar Representações
# BoW
bow = CountVectorizer()
X_bow = bow.fit_transform(textos)

# TF-IDF
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(textos)

# Embeddings (Multilíngue para português)
modelo = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
X_emb = modelo.encode(textos)

print("Representações geradas com sucesso!")


In [ ]:
# 3. Análise dos Pares Específicos
def calcular_similaridades(id1, id2):
    idx1 = ids.index(id1)
    idx2 = ids.index(id2)
    
    sim_bow = cosine_similarity(X_bow[idx1], X_bow[idx2])[0][0]
    sim_tfidf = cosine_similarity(X_tfidf[idx1], X_tfidf[idx2])[0][0]
    sim_emb = cosine_similarity(X_emb[idx1].reshape(1, -1), X_emb[idx2].reshape(1, -1))[0][0]
    
    print(f"\nComparando: [{id1}] e [{id2}]")
    print(f"Texto 1: '{textos[idx1]}'")
    print(f"Texto 2: '{textos[idx2]}'")
    print(f"  BoW:    {sim_bow:.3f}")
    print(f"  TF-IDF: {sim_tfidf:.3f}")
    print(f"  Embed:  {sim_emb:.3f}")

# Caso 1: Mesma semântica, palavras diferentes
calcular_similaridades("M003", "M017")

# Caso 2: Mesma semântica, palavras diferentes (Saúde)
calcular_similaridades("M008", "M022")

# Caso 3: Temas completamente diferentes, mas curtos
calcular_similaridades("M008", "M031")


### Conclusão
- **Limitações do BoW e TF-IDF:** Ambas são representações lexicais estritas. Nos pares M003xM017 e M008xM022 (onde os cidadãos relatam o mesmo problema usando vocabulários diferentes), o BoW e TF-IDF entregam similaridade 0.000, falhando completamente em detectar a redundância. Eles dependem da correspondência exata de palavras.
- **Vantagens dos Embeddings:** O modelo `paraphrase-multilingual` mapeia o significado para um espaço vetorial. Ele conseguiu identificar alta similaridade entre "buraco enorme" e "esburacado" (M003xM017) e entre "sem médico" e "falta atendimento" (M008xM022). Ao mesmo tempo, ele corretamente atribuiu uma similaridade baixa ao comparar "posto sem médico" com "lâmpada queimada" (M008xM031), demonstrando que compreende a diferença de contexto.
